# Script 13 - Linearity of the rhythm-outcome association (Supp Table S10, Supp Fig S3)

Fits each rhythm metric against each outcome twice -- once linear, once with a
natural cubic spline (df = 4) -- and Wald-tests the non-linear basis columns.
A rejection means the spline fits better than a straight line; it does not
identify a threshold.

All three metrics are tested against PHQ-9 (18+) and the cognition composite
(60+). Estimates are WLS-HC3 without the complex-survey design correction; the
notebook checks the linearity assumption of the Mplus models and does not
re-estimate effects.

Reads `data/processed/8_nhanes_analytic.csv`. Writes
`result/supplementary/SuppTableS10_GAM_Linearity.{csv,xlsx}`,
`result/raw/script13_GAM_full.csv`,
`figure/supplementary/SuppFigS3_GAM_Partial_Effects.{pdf,png}` and
`figure/source_data/Source_Data_SuppFigS3_GAM_Partial_Effects.xlsx`.


## Step 0 — Imports and paths

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 60

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE   = find_root()
PROC   = BASE / 'data' / 'processed'
R_SUPP = BASE / 'result' / 'supplementary'
R_RAW  = BASE / 'result' / 'raw'
F_SUPP = BASE / 'figure' / 'supplementary'
F_SRC  = BASE / 'figure' / 'source_data'
for q in (R_SUPP, R_RAW, F_SUPP, F_SRC):
    q.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook runs against the code on disk.
import importlib
import wls_utils, nature_palette
for _m in (wls_utils, nature_palette,):
    importlib.reload(_m)

# chi2_sf comes from wls_utils; scipy is not a dependency.
from wls_utils import wls_fit, ns_basis, smooth_pdep, chi2_sf
from nature_palette import PHQ9_COLOR, COG_COLOR, GREY
from nature_palette import fmt_ci as _fmt_ci, age_label

from nature_palette import EXP_COL, EXP_ORDER  # one source of truth
EXPOSURES = list(EXP_ORDER)
EXP_NAME  = {'IV': 'Intradaily variability', 'IS': 'Interdaily stability',
             'RA': 'Relative amplitude'}


## Step 1 — Load + rename + derive PHQ9 / COG_Z

In [ ]:
df = pd.read_csv(PROC / '8_nhanes_analytic.csv', low_memory=False)
rename = {
    'RIDAGEYR':       'AGE',
    'RIAGENDR':       'SEX',
    'BMXBMI':         'BMI',
    'DMDEDUC2':       'EDU',
    'Wear_Time':      'WEAR_T',
    'SLD010H':        'SLEEP_H',
}
df = df.rename(columns=rename)
assert 'WTMEC4YR' in df.columns, 'run script 8: WTMEC4YR missing'
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'
df = df[df['AGE'] >= 18].copy()

dpq_cols = [f'DPQ{i:03d}' for i in [10, 20, 30, 40, 50, 60, 70, 80, 90]]
df['PHQ9'] = df[dpq_cols].replace({7: np.nan, 9: np.nan}).sum(axis=1, min_count=9)

elig_60 = df['AGE'] >= 60
for src, dst in [('memory_score', 'MEM_Z'),
                 ('fluency_score', 'FLUEN_Z'),
                 ('speed_score',  'SPEED_Z')]:
    raw = df[src].replace(0, np.nan)
    s = raw.loc[elig_60].dropna()
    df[dst] = np.where(elig_60, (raw - s.mean()) / s.std(), np.nan)
df['COG_Z'] = df[['MEM_Z', 'FLUEN_Z', 'SPEED_Z']].mean(axis=1, skipna=False)

print(f'Adults 18+: n = {len(df):,}')
print(f'  PHQ-9 non-null: {df["PHQ9"].notna().sum():,}')
print(f'  COG_Z non-null (60+): {df["COG_Z"].notna().sum():,}')

## Step 2 — Helpers: per-outcome sample + survey-weighted standardisation

In [ ]:
M1_COVS = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']

def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def fmt_ci(est, lo, hi, dp=3):
    return _fmt_ci(est, lo, hi, dp)   # minus kept, plus omitted

def weighted_mean_sd(x, w):
    w_tot = np.sum(w)
    mu = np.sum(x * w) / w_tot
    var = np.sum(w * (x - mu) ** 2) / w_tot
    return mu, float(np.sqrt(var))

def per_outcome_sample(exposure, outcome_col, age_min=18):
    needed = [exposure, outcome_col, 'WTMEC4YR'] + M1_COVS
    mask = (df['AGE'] >= age_min) & df[needed].notna().all(axis=1)
    return df.loc[mask].copy()


## Step 3 — Fit linear + GAM, Wald chi-square nonlinearity test

For each metric and outcome:
1. Standardise the metric and the outcome using survey-weighted SDs on the sample.
2. Linear model: covariates + metric_z.
3. GAM: covariates + ns_basis(metric_z, df = 4) (4 columns = 1 linear + 3 smooth).
4. Wald χ² test on the three smooth basis coefficients.
5. GAM partial-dependence curve on a grid from the 1st to the 99th percentile of the metric.

In [ ]:
def fit_linear_and_gam(exposure, outcome_col, age_min=18, df_spline=4):
    """Linear vs natural-cubic-spline fit for one rhythm metric and outcome; a
    rejection means the spline fits better than a straight line, not that a
    threshold exists."""
    ana = per_outcome_sample(exposure, outcome_col, age_min=age_min)
    w = ana['WTMEC4YR'].values
    mu_x, sd_x = weighted_mean_sd(ana[exposure].values,    w)
    mu_y, sd_y = weighted_mean_sd(ana[outcome_col].values, w)
    x_z = (ana[exposure].values    - mu_x) / sd_x
    y_z = (ana[outcome_col].values - mu_y) / sd_y
    cov_mat = np.column_stack([ana[c].values for c in M1_COVS])

    X_lin = np.column_stack([x_z, cov_mat])
    fit_lin = wls_fit(X_lin, y_z, w=w, se='hc3', add_const=True)

    B_x = ns_basis(x_z, df=df_spline)
    X_gam = np.column_stack([B_x, cov_mat])
    fit_gam = wls_fit(X_gam, y_z, w=w, se='hc3', add_const=True)

    # Wald test on the non-linear basis columns (cols 2..df_spline; col 1 is
    # the linear direction inside the basis).
    smooth_idx = list(range(2, 1 + df_spline))
    beta_s = fit_gam['params'][smooth_idx]
    cov_s = fit_gam['cov'][np.ix_(smooth_idx, smooth_idx)]
    W = float(beta_s.T @ np.linalg.pinv(cov_s) @ beta_s)
    df_smooth = len(smooth_idx)
    p_nonlin = chi2_sf(W, df_smooth)

    grid_raw = np.linspace(np.percentile(ana[exposure], 1),
                           np.percentile(ana[exposure], 99), 200)
    grid_z = (grid_raw - mu_x) / sd_x
    pooled = np.concatenate([x_z, grid_z])
    B_grid = ns_basis(pooled, df=df_spline)[len(x_z):]
    x_idx = list(range(1, 1 + df_spline))
    f_pdep, lo_pdep, hi_pdep = smooth_pdep(grid_raw, fit_gam['params'][x_idx],
                                           B_grid, grid_raw,
                                           fit_gam['cov'], x_idx)
    # Centre the spline curve to zero at the sample-mean exposure, where the
    # linear reference is zero by construction; the Wald test is invariant to
    # this vertical shift.
    off = float(np.interp(0.0, grid_z, f_pdep))
    f_pdep, lo_pdep, hi_pdep = f_pdep - off, lo_pdep - off, hi_pdep - off
    return {
        'exposure': exposure, 'outcome': outcome_col, 'n': int(fit_lin['n']),
        'beta_linear': float(fit_lin['params'][1]),
        'ci_lo_linear': float(fit_lin['ci_lo'][1]),
        'ci_hi_linear': float(fit_lin['ci_hi'][1]),
        'p_linear': float(fit_lin['p'][1]),
        'r2_linear': float(fit_lin['r2']), 'r2_gam': float(fit_gam['r2']),
        'delta_r2': float(fit_gam['r2'] - fit_lin['r2']),
        'wald_chi2': W, 'df_smooth': df_smooth, 'p_nonlin': p_nonlin,
        'grid_raw': grid_raw, 'grid_z': grid_z,
        'f_pdep': f_pdep, 'lo_pdep': lo_pdep, 'hi_pdep': hi_pdep,
        'f_lin': float(fit_lin['params'][1]) * grid_z,
    }


## Step 4 — Run for Depression and Cognition

In [ ]:
# Every rhythm metric, both outcomes.
RES = {}
for exp in EXPOSURES:
    RES[(exp, 'PHQ9')]  = fit_linear_and_gam(exp, 'PHQ9',  age_min=18)
    RES[(exp, 'COG_Z')] = fit_linear_and_gam(exp, 'COG_Z', age_min=60)

for (exp, out), R in RES.items():
    lab = f'{EXP_NAME[exp]} -> ' + ('PHQ-9 (\u226518)' if out == 'PHQ9' else 'cognition (60+)')
    verdict = 'NON-LINEAR' if R['p_nonlin'] < 0.05 else 'linear supported'
    print(f'{lab:52s} n={R["n"]:>6,}  linear beta={R["beta_linear"]:.3f} '
          f'(p={fmt_p(R["p_linear"])})   Wald chi2({R["df_smooth"]})={R["wald_chi2"]:6.2f} '
          f'p_nonlin={fmt_p(R["p_nonlin"])}  -> {verdict}')


## Step 5 — Build Supp Table S10

In [ ]:
rows = []
for (exp, out), R in RES.items():
    rows.append({
        'Rhythm metric': f'{EXP_NAME[exp]} ({exp})',
        'Outcome': 'PHQ-9 (\u226518)' if out == 'PHQ9' else 'Cognition composite (\u226560)',
        'n': f"{R['n']:,}",
        'Linear beta (95% CI)': fmt_ci(R['beta_linear'], R['ci_lo_linear'], R['ci_hi_linear']),
        'Linear p': fmt_p(R['p_linear']),
        'R2 linear': f"{R['r2_linear']:.4f}",
        'R2 GAM': f"{R['r2_gam']:.4f}",
        'delta R2': f"{R['delta_r2']:.5f}",
        'Wald chi2 (df=3)': f"{R['wald_chi2']:.3f}",
        'p (non-linearity)': fmt_p(R['p_nonlin']),
        'Verdict': 'NON-LINEAR' if R['p_nonlin'] < 0.05 else 'linear OK',
    })
tableS10 = pd.DataFrame(rows)
print(tableS10.to_string(index=False))
tableS10.to_csv(R_SUPP / 'SuppTableS10_GAM_Linearity.csv', index=False)
pd.DataFrame([{k: v for k, v in R.items()
               if not isinstance(v, np.ndarray)} for R in RES.values()]
             ).to_csv(R_RAW / 'script13_GAM_full.csv', index=False)

NOTES = [
    'Survey-weighted WLS with HC3 errors; NO complex-survey design correction. '
    'Inference follows the Mplus models.',
    'GAM: natural cubic spline basis (df = 4) on the z-standardised metric; '
    'Wald test on the non-linear basis columns.',
    'A significant test means the spline fits better than a straight line. It '
    'does not identify a threshold.',
    'Covariates: age, sex, BMI, education, wear time, sleep duration.',
]
with pd.ExcelWriter(R_SUPP / 'SuppTableS10_GAM_Linearity.xlsx',
                    engine='openpyxl') as xw:
    tableS10.to_excel(xw, sheet_name='Supp Table S10', index=False)
    pd.DataFrame({'Notes': NOTES}).to_excel(xw, sheet_name='Notes', index=False)
print(f"\nwrote {R_SUPP / 'SuppTableS10_GAM_Linearity.xlsx'}")

for stale in [R_SUPP / 'SuppTableS5_GAM_Linearity.csv',
              R_SUPP / 'SuppTableS5_GAM_Linearity.xlsx']:
    if stale.exists():
        dest = BASE / '_superseded' / stale.name
        dest.parent.mkdir(exist_ok=True)
        stale.rename(dest)
        print(f'superseded -> _superseded/{stale.name}')


## Step 6 — Supp Figure S3: GAM partial-effect plots

In [ ]:
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 7, 'axes.titlesize': 7, 'axes.labelsize': 6,
    'xtick.labelsize': 6, 'ytick.labelsize': 6, 'legend.fontsize': 5.5,
    'axes.linewidth': 0.5, 'lines.linewidth': 0.9,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 600,
    'axes.spines.top': False, 'axes.spines.right': False,
})
MM = 1 / 25.4

fig, axes = plt.subplots(2, 3, figsize=(183 * MM, 105 * MM), sharex='col')
for j, exp in enumerate(EXPOSURES):
    for i, (out, olab, ocol) in enumerate([
            ('PHQ9', 'PHQ-9 (\u226518)', PHQ9_COLOR),
            ('COG_Z', 'Cognition (60+)', COG_COLOR)]):
        ax = axes[i, j]
        R = RES[(exp, out)]
        ax.fill_between(R['grid_raw'], R['lo_pdep'], R['hi_pdep'],
                        color=EXP_COL[exp], alpha=0.15, lw=0)
        ax.plot(R['grid_raw'], R['f_pdep'], color=EXP_COL[exp], lw=1.1)
        ax.plot(R['grid_raw'], R['f_lin'], color=GREY, lw=0.8, ls='--')
        ax.axhline(0, color='0.4', lw=0.4, ls=':')
        ax.text(0.03, 0.94,
                f"p(non-lin) = {fmt_p(R['p_nonlin'])}",
                transform=ax.transAxes, va='top', fontsize=5.5)
        if i == 0:
            ax.set_title(f'{EXP_NAME[exp]} ({exp})', color=EXP_COL[exp], pad=3)
        if i == 1:
            ax.set_xlabel(f'{exp} (raw scale)')
        if j == 0:
            ax.set_ylabel(f'Partial effect on {olab} (z)', color=ocol)
fig.tight_layout()

stem = 'SuppFigS3_GAM_Partial_Effects'
fig.savefig(F_SUPP / f'{stem}.pdf', bbox_inches='tight')
fig.savefig(F_SUPP / f'{stem}.png', bbox_inches='tight', dpi=600)
plt.close(fig)
print(f'wrote {F_SUPP / stem}.pdf + .png')

cap = ['Supplementary Fig. S3 | Linearity of each rhythm-outcome association.',
       'Solid: GAM partial effect (natural cubic spline, df = 4) with 95% CI. '
       'Dashed grey: the linear fit.',
       'A significant test means the spline beats a straight line; it does not '
       'identify a threshold.',
       'Both curves are centred to zero at the sample-mean exposure, so they '
       'are comparable in shape, not in level.',
       'WLS-HC3, no design correction; inference follows Mplus.']
with pd.ExcelWriter(F_SRC / f'Source_Data_{stem}.xlsx', engine='openpyxl') as xw:
    for (exp, out), R in RES.items():
        sheet = f"{exp} {'PHQ9' if out == 'PHQ9' else 'COG'}"
        pd.DataFrame({
            f'{exp}_raw': R['grid_raw'],
            'GAM_partial_effect': R['f_pdep'],
            'GAM_CI_lo': R['lo_pdep'], 'GAM_CI_hi': R['hi_pdep'],
            'Linear_reference': R['f_lin'],
        }).to_excel(xw, sheet_name=sheet, index=False)
    pd.DataFrame({'Notes': cap}).to_excel(xw, sheet_name='legend', index=False)
print(f'wrote {F_SRC / ("Source_Data_" + stem + ".xlsx")}')

for stale in [F_SUPP / f'{stem}.tiff', F_SRC / 'Source_Data_SuppFigS3.xlsx']:
    if stale.exists():
        dest = BASE / '_superseded' / stale.name
        dest.parent.mkdir(exist_ok=True)
        stale.rename(dest)
        print(f'superseded -> _superseded/{stale.name}')
